# Uber Fare Prediction — Polynomial Regression

**Objective**
1. Understand the dataset & clean it up.
2. Build regression models (Linear → Polynomial) to predict `fare_amount`.
3. Evaluate and compare models using R², RMSE and MAE.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Understand the dataset

In [ ]:
df = pd.read_csv("uber.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print("Missing values:\n", df.isna().sum(), "\n")
print("fare_amount <= 0      :", (df["fare_amount"] <= 0).sum())
print("passenger_count == 0  :", (df["passenger_count"] == 0).sum())
print("passenger_count > 6   :", (df["passenger_count"] > 6).sum())
print("zero coordinates      :", ((df["pickup_longitude"] == 0) | (df["dropoff_longitude"] == 0)).sum())
print("latitude out of range :", ((df["pickup_latitude"].abs() > 90) | (df["dropoff_latitude"].abs() > 90)).sum())

**Observations**
- `Unnamed: 0` and `key` are identifiers — no predictive value.
- One row has missing dropoff coordinates.
- Negative/zero fares, 0 or 208 passengers, and coordinates at (0, 0) or outside valid lat/long ranges are data-entry errors.
- The data is NYC taxi trips, so coordinates should fall inside a NYC bounding box.

## 2. Cleanup

In [ ]:
data = df.drop(columns=["Unnamed: 0", "key"]).dropna().copy()
n0 = len(data)

# NYC bounding box (generous, includes airports)
LON_MIN, LON_MAX = -74.3, -73.7
LAT_MIN, LAT_MAX = 40.5, 41.0
in_nyc = (
    data["pickup_longitude"].between(LON_MIN, LON_MAX)
    & data["dropoff_longitude"].between(LON_MIN, LON_MAX)
    & data["pickup_latitude"].between(LAT_MIN, LAT_MAX)
    & data["dropoff_latitude"].between(LAT_MIN, LAT_MAX)
)
data = data[in_nyc]
data = data[data["passenger_count"].between(1, 6)]
data = data[data["fare_amount"].between(2.5, 200)]   # $2.50 is the NYC flag-drop fare
print(f"Rows: {n0} -> {len(data)}  (removed {n0 - len(data)})")

### Feature engineering
- **distance_km**: haversine (great-circle) distance between pickup and dropoff — the main fare driver.
- **Time features** from `pickup_datetime`: year (fares rose over time), hour, day of week, month.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

data["distance_km"] = haversine_km(
    data["pickup_latitude"], data["pickup_longitude"],
    data["dropoff_latitude"], data["dropoff_longitude"],
)
dt = pd.to_datetime(data["pickup_datetime"], utc=True)
data["year"] = dt.dt.year
data["month"] = dt.dt.month
data["hour"] = dt.dt.hour
data["day_of_week"] = dt.dt.dayofweek

# Remove zero-distance trips (same pickup/dropoff) and implausibly long ones
n1 = len(data)
data = data[data["distance_km"].between(0.1, 60)]
print(f"Rows: {n1} -> {len(data)}  (removed {n1 - len(data)} zero/extreme distance trips)")
data[["fare_amount", "distance_km", "passenger_count", "year", "month", "hour", "day_of_week"]].describe().T

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(data["fare_amount"], bins=100)
axes[0].set(title="Fare distribution", xlabel="fare ($)", ylabel="trips", xlim=(0, 80))
axes[1].scatter(data["distance_km"], data["fare_amount"], s=1, alpha=0.2)
axes[1].set(title="Fare vs distance", xlabel="distance (km)", ylabel="fare ($)")
data.groupby("year")["fare_amount"].mean().plot(ax=axes[2], marker="o")
axes[2].set(title="Mean fare by year", ylabel="fare ($)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data.groupby("hour")["fare_amount"].mean().plot(ax=axes[0], marker="o")
axes[0].set(title="Mean fare by hour of day", ylabel="fare ($)")
data.groupby("day_of_week")["fare_amount"].mean().plot(ax=axes[1], kind="bar")
axes[1].set(title="Mean fare by day of week (0 = Mon)", ylabel="fare ($)")
plt.tight_layout()
plt.show()

In [ ]:
FEATURES = ["distance_km", "passenger_count", "year", "month", "hour", "day_of_week",
            "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"]
TARGET = "fare_amount"
data[FEATURES + [TARGET]].corr()[TARGET].sort_values(ascending=False)

**Observations**
- Fare is strongly (and fairly linearly) related to distance, but the scatter shows curvature and clusters
  (e.g. flat-rate JFK trips around $45–57) — a reason to try polynomial terms.
- Fares increase over the years (fare hikes in 2012).
- Hour and day of week have smaller, non-linear effects — again suited to polynomial / interaction terms.

## 4. Train / test split

In [ ]:
X = data[FEATURES]
y = data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("train:", X_train.shape, " test:", X_test.shape)

## 5. Models

Polynomial regression = linear regression on polynomial-expanded features
(`x1, x2, x1², x1·x2, x2², ...`). Pipeline per model:

`PolynomialFeatures(degree)` → `StandardScaler` → `LinearRegression` / `Ridge`

Scaling after expansion keeps the high-order terms on comparable scales; Ridge adds L2 regularisation to
tame the many correlated terms at higher degrees.

First, choose the degree with 5-fold cross-validation on the training set.

In [ ]:
def poly_model(degree, alpha=None):
    reg = LinearRegression() if alpha is None else Ridge(alpha=alpha)
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), reg)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for degree in [1, 2, 3, 4]:
    model = poly_model(degree, alpha=1.0)
    scores = cross_val_score(model, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1)
    n_terms = PolynomialFeatures(degree, include_bias=False).fit(X_train.iloc[:5]).n_output_features_
    cv_rows.append({"degree": degree, "n_terms": n_terms, "cv_rmse_mean": -scores.mean(), "cv_rmse_std": scores.std()})
    print(f"degree={degree}  terms={n_terms:4d}  CV RMSE={-scores.mean():.3f} ± {scores.std():.3f}")
cv_df = pd.DataFrame(cv_rows)

plt.figure(figsize=(6, 4))
plt.errorbar(cv_df["degree"], cv_df["cv_rmse_mean"], yerr=cv_df["cv_rmse_std"], marker="o", capsize=4)
plt.xticks(cv_df["degree"])
plt.title("5-fold CV RMSE vs polynomial degree (Ridge α=1)")
plt.xlabel("degree"); plt.ylabel("RMSE ($)")
plt.show()

### Fit final models and evaluate on the held-out test set

In [ ]:
best_degree = int(cv_df.loc[cv_df["cv_rmse_mean"].idxmin(), "degree"])
print("Best degree by CV:", best_degree)

models = {
    "Linear Regression (distance only)": (make_pipeline(LinearRegression()), ["distance_km"]),
    "Linear Regression (all features)":  (poly_model(1), FEATURES),
    "Polynomial deg 2":                  (poly_model(2), FEATURES),
    "Polynomial deg 3":                  (poly_model(3), FEATURES),
    f"Polynomial deg {best_degree} + Ridge (best CV)": (poly_model(best_degree, alpha=1.0), FEATURES),
}

def evaluate(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
    }

rows, preds = [], {}
for name, (model, cols) in models.items():
    model.fit(X_train[cols], y_train)
    p_train = model.predict(X_train[cols])
    p_test = model.predict(X_test[cols])
    preds[name] = p_test
    tr, te = evaluate(y_train, p_train), evaluate(y_test, p_test)
    rows.append({"model": name,
                 "train_R2": tr["R2"], "test_R2": te["R2"],
                 "train_RMSE": tr["RMSE"], "test_RMSE": te["RMSE"],
                 "test_MAE": te["MAE"]})

results = pd.DataFrame(rows).set_index("model").sort_values("test_RMSE")
results

## 6. Model comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
r = results.sort_values("test_RMSE", ascending=False)
axes[0].barh(r.index, r["test_R2"])
axes[0].set(title="Test R² (higher is better)", xlim=(0, 1))
axes[1].barh(r.index, r["test_RMSE"])
axes[1].set(title="Test RMSE in $ (lower is better)")
axes[1].set_yticklabels([])
for ax, col in zip(axes, ["test_R2", "test_RMSE"]):
    for i, v in enumerate(r[col]):
        ax.text(v, i, f" {v:.3f}", va="center")
plt.tight_layout()
plt.show()

In [ ]:
best_name = results.index[0]
best_pred = preds[best_name]
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, best_pred, s=2, alpha=0.2)
lim = [0, 100]
axes[0].plot(lim, lim, "r--", lw=1)
axes[0].set(title=f"Actual vs predicted — {best_name}", xlabel="actual fare ($)", ylabel="predicted fare ($)",
            xlim=lim, ylim=lim)
axes[1].hist(residuals, bins=100, range=(-30, 30))
axes[1].set(title="Residuals (actual − predicted)", xlabel="$", ylabel="trips")
plt.tight_layout()
plt.show()

## 7. Predict a new trip

In [ ]:
best_model, best_cols = models[best_name]
new_trip = pd.DataFrame([{
    "pickup_longitude": -73.9855, "pickup_latitude": 40.7580,    # Times Square
    "dropoff_longitude": -73.9654, "dropoff_latitude": 40.7829,  # Central Park
    "passenger_count": 2,
    "pickup_datetime": "2015-06-15 18:30:00",
}])
new_trip["distance_km"] = haversine_km(new_trip["pickup_latitude"], new_trip["pickup_longitude"],
                                       new_trip["dropoff_latitude"], new_trip["dropoff_longitude"])
t = pd.to_datetime(new_trip["pickup_datetime"])
new_trip["year"], new_trip["month"], new_trip["hour"], new_trip["day_of_week"] = t.dt.year, t.dt.month, t.dt.hour, t.dt.dayofweek
print(f"Distance: {new_trip['distance_km'].iloc[0]:.2f} km")
print(f"Predicted fare: ${best_model.predict(new_trip[best_cols])[0]:.2f}")

## 8. Conclusions

- **Cleanup** removed missing rows, invalid fares, invalid passenger counts, coordinates outside NYC and
  zero-distance trips — essential, since raw outliers (e.g. $499 fares, latitude 1644) wreck a least-squares fit.
- **Distance** is by far the strongest predictor; a one-feature linear model already explains most of the variance.
- **Polynomial features** capture curvature and interactions (distance × year for fare hikes, coordinates for
  airport flat rates, time-of-day effects), lowering RMSE versus plain linear regression.
- Gains flatten (and can reverse) at higher degrees as the term count explodes — cross-validation plus Ridge
  regularisation picks a degree that improves fit without overfitting (train and test scores stay close).